

**Objective:** calculate ANPP
Calculate ANPP in the following way:
1. Eddy flux measures NEE (CO2 input - output of whole system)
2. Daytime or Nighttime model calculates RECO (CO2 out of whole system) and GPP (CO2 in of whole ecosystem, only autotrophes consume CO2 so should be 100% autotrophic) from NEE at ICOS
3. Using measurements I average the soil R for each year
4. Based on measurements derived ratio I estimate how much of soil R is heterotrophic (this should be the only heterotrophic component of NEE)
5. I subtract herotrophic soil R to NEE (soil R is the only heterotrophic component of NEE and releases carbon from the system, subtracting  soil R from NEE should theoretically arrive at ANPP)

Code cells & steps:

**ICOS data retriever**
1. Download ICOS dataset
2. Select relevant parameters
3. Write to file and pass on path

**Soil respiration data retriever**
1. Download & clean soil respiration dataset
2. Calculate monthly averages
3. Fill in empty months using average for the respective month
4. Calculate average per year
5. Write to file and pass on path

**Soil trophic data retreiver**
data appears incongruent with other data so this is currently skipped and substituted with a formula
1. Download & clean soil trophic dataset
2. (resolve dataset specific issues)
3. Calculate heterotrophic respiration based on trenched vs non-trenched
4. Calculate average per year
5. Write to file and pass on path


**ANPP calculator**
1. Read outputs from previous cells
2. Join datasets by year
3. Derive soil R (heterotrophic / soil R) ratio from soil trophic data for each year, and add to NEE & RECO dataframe **(currently data validity is in question so use formula from https://onlinelibrary.wiley.com/doi/full/10.1111/j.1365-2486.2004.00816.x )**
4. Calculate new column in dataframe for ANPP using formula: ANPP = NEE - (soil R * soil R ratio) **(with current solution soil R is calculated in step 3 and can be added directly)** 


**Objective 2:** calibration
1. Define parameters to test
2. run model with each combination of parameters
3. compare ANPP outcome of model to ANPP from observations

In [ ]:
#Params (WIP)
#=============================================
#years to compute npp for (must be within data limit)


In [ ]:
#Install and load packages
#=============================================
if (!require("pacman")) install.packages("pacman")
pacman::p_load("dplyr", "tidyr", "readxl", "readr", "lubridate")

In [ ]:
#ICOS data retriever
#=============================================
# Part 1: Retrieve ICOS data
#=============================================
#possibly temporary
#once a more permanent solution has been selected, filepath should be an input variable that is an output from the python cell or importing should be done in this cell
#Read the file retrieved by python importer
ICOS_fluxes_yearly <- read.csv("./icos_data/ICOS_NL-Loo_FLUXNET_FLUXMET_YY_1997-2026_v1.3_r1.csv")

#Selecting relevant parameters
ICOS_fluxes_yearly <- ICOS_fluxes_yearly %>%
    select(`TIMESTAMP`, `RECO_DT_CUT_REF`, `RECO_NT_CUT_REF`, `NEE_CUT_REF`) %>% #Select relevant columns, NT is currently not used but included to compare results
    subset(RECO_DT_CUT_REF > -9999) #Remove years with no data
#in yearly dataset, RECO and NEE are in gC m-2 y-1

#Save data as csv to pass on to next cell
ICOS_fluxes_yearly_path <- "ICOS_fluxes_yearly.csv"

write.csv(ICOS_fluxes_yearly, paste0("/tmp/data/", ICOS_fluxes_yearly_path), row.names=FALSE)

In [ ]:
#Soil respiration data retriever
#=============================================
# Part 1: Retrieve soil respiration data
#=============================================
#Download and read file
path <- "/tmp/data/NL-Loo_soilresp_data.xlsx"
url <- "https://zenodo.org/records/15721310/files/NL-Loo_soilresp_data.xlsx?download=1"

if(!file.exists(path)){
    download.file(url, path, mode = "wb", sheet_name="Data all")
}

soil_r_data <- read_excel(path)

#Remove spaces from relevant variable names
soil_r_data <- rename(soil_r_data, `soil_r_flux_(µmol_m-2_s-1)` = `soil_respiration_flux (µmol m-2 s-1)`)

#Remove blank rows
soil_r_data <- filter(soil_r_data,, !is.na(`soil_r_flux_(µmol_m-2_s-1)`))

#should maybe filter to just daytime measurements???

#=============================================
# Part 2: Average per year
#=============================================
#Calculate monthly average for months with data
soil_r_data <- soil_r_data %>%
    group_by(year, month) %>%
    summarise(
        avg_soil_flux = mean(`soil_r_flux_(µmol_m-2_s-1)`, na.rm = TRUE),
        .groups = "drop"
    )

#Calculate average per month over several years (each month has equal weight irrespective of amount of measurements)
month_avg_soil_r <- soil_r_data %>%
    group_by(month) %>%
    summarise(
        month_avg = mean(avg_soil_flux, na.rm= TRUE),
        .groups = "drop"
        )

#Add monthly averages to fill months without data
soil_r_data <- soil_r_data %>%
    complete(year, month =1:12) %>% #Make sure there is an entry for every month, even if empty
    left_join(month_avg_soil_r, by = "month") %>%
    mutate(avg_flux = coalesce(avg_soil_flux, month_avg)) %>% #find the empty values in avg_flux and replace them with the corresponding month average
    group_by(year) %>%
    summarise(
        yearly_avg_soil_r = mean(avg_flux),
        .groups = "drop"
    ) %>%
    select(year, yearly_avg_soil_r) #select relevant columns

#how to make it an output?
soil_r_yearly_path <- "preprocessed_soil_r_yearly.csv"

write.csv(soil_r_data, paste0("/tmp/data/", soil_r_yearly_path), row.names=FALSE)

In [ ]:
#ANPP calculator
#=============================================
# Part 1: Retrieve respiration data from previous cells
#=============================================
soil_r_yearly <- read.csv(paste0("/tmp/data/", soil_r_yearly_path))
ICOS_fluxes_yearly <- read.csv(paste0("/tmp/data/", ICOS_fluxes_yearly_path))

#Change 'TIMESTAMP' to 'year' so dataset share colum name for same variable
colnames(ICOS_fluxes_yearly)[colnames(ICOS_fluxes_yearly) == 'TIMESTAMP'] <- 'year'

#Merge datasets by year (retaining the years that occur in both datasets)
combined_data <- left_join(soil_r_yearly, ICOS_fluxes_yearly, by ='year')

#=============================================
# Part 1: Calculating ANPP
#=============================================
#RECO and NEE are in gC m-2 y-1 while soil_r is in µmol CO2 m-2 s-1
#Tmporary relationship between total soil_r and heterotrophic soil soil_r: ln (RH)=1.22+0.73 ln(RS)

#need to change the if statement, does not work like this inside mutate
ANPP_data <- combined_data %>%
    mutate(
        aggregated_soil_r = if_else(
    leap_year(year),
    (yearly_avg_soil_r * 31622400 * 12.0111)/1000000, #multiply by seconds in a leap year to go to year and multiply by molecular mass of carbon to go to µg C, finally devide by 1000000 to go to g C 
    (yearly_avg_soil_r * 31536000 * 12.0111)/1000000), #multiply by seconds in a non-leap year to go to year and multiply by molecular mass of carbon to go to µg C, finally devide by 1000000 to go to g C
        soil_r_het = exp(1.22 + 0.73 * log(aggregated_soil_r)), #apply formula from https://onlinelibrary.wiley.com/doi/full/10.1111/j.1365-2486.2004.00816.x to arrive at soil_r_het from soil_r
        ANPP = (NEE_CUT_REF - soil_r_het) * -1 #add heterotrophic respiration back to NEE to arrive at ANPP, multiply with -1 as carbon storage in NEE is negative but positive in ANPP
  )

ANPP_data
#soil r is higher than RECO??? look up  realistic values for soil r, RECO, soil r het & ANPP